# Coding Assistant: Tool-Assisted Context Retrieval for Code Q&A

A prototype of a **coding assistant agent** that answers developer questions by:
1. Classifying the query intent (lookup / explain / debug / generate)
2. Selecting and calling the right retrieval tools
3. Building a ranked context window from the results
4. Synthesizing a structured code-focused answer with source citations

---

## Architecture

```text
                       ┌──────────────────────┐
  User Query ────────► │  Intent Classifier   │
                       └──────────┬───────────┘
                                  │ intent + keywords
                       ┌──────────▼───────────┐
                       │   Tool Router        │ decides which tools to call
                       └──────────┬───────────┘
              ┌───────────────────┼───────────────────┐
              │                   │                   │
   ┌──────────▼───────┐ ┌────────▼────────┐ ┌────────▼────────┐
   │ search_docs      │ │ search_snippets │ │ lookup_function │
   │ (doc fragments)  │ │ (code patterns) │ │ (signatures)    │
   └──────────┬───────┘ └────────┬────────┘ └────────┬────────┘
              └───────────────────┼───────────────────┘
                                  │ retrieved context pieces
                       ┌──────────▼───────────┐
                       │  Context Builder     │ dedup, rank, trim
                       └──────────┬───────────┘
                                  │ ranked context window
                       ┌──────────▼───────────┐
                       │  Answer Synthesizer  │ template + code gen
                       └──────────┬───────────┘
                                  │
                         Structured Answer
                    (explanation + code + citations)
```

**Tool inventory:**

| Tool | Layer | Purpose |
|---|---|---|
| `search_docs` | Retrieval | Full-text search over doc fragments |
| `search_snippets` | Retrieval | Fuzzy search over indexed code patterns |
| `lookup_function` | Lookup | Exact function signature + docstring |
| `get_examples` | Retrieval | Runnable code examples by topic |
| `search_repo` | Retrieval | Simulate searching a codebase by pattern |
| `run_code` | Execution | Safely execute a code snippet and return output |


## 1. Setup

In [ ]:
!pip install -q pandas matplotlib

In [ ]:
import ast
import json
import math
import re
import textwrap
import traceback
from collections import defaultdict
from dataclasses import dataclass, field, asdict
from datetime import datetime
from enum import Enum
from io import StringIO
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import pandas as pd

ARTIFACT_DIR = Path.cwd() / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

print("Setup complete.")

## 2. Knowledge Base — Documentation Fragments

The assistant's knowledge base contains three layers of indexed content:
1. **Doc fragments** — prose explanations from official docs (Python, pandas, requests, etc.)
2. **Code snippets** — canonical patterns for common tasks
3. **Repo stubs** — simulated files from a codebase (for `search_repo`)


In [ ]:
# ── Documentation fragments ──────────────────────────────────────────────────
# Each entry: { "id", "library", "topic", "keywords": [...], "content" }
DOC_FRAGMENTS = [
    {
        "id": "py-list-01", "library": "python", "topic": "list", "version": "3.12",
        "keywords": ["list", "append", "extend", "insert", "remove", "sort", "reverse", "comprehension"],
        "content": (
            "Lists are mutable sequences used to store collections of items. "
            "Key operations: list.append(x) adds a single item to the end; "
            "list.extend(iterable) appends all items from an iterable; "
            "list.insert(i, x) inserts before position i; "
            "list.remove(x) removes the first occurrence of x; "
            "list.sort(key=None, reverse=False) sorts in-place; "
            "list.reverse() reverses in-place. "
            "List comprehensions: [expr for item in iterable if condition] are the idiomatic way "
            "to build lists. They are faster than map/filter for most workloads."
        ),
    },
    {
        "id": "py-dict-01", "library": "python", "topic": "dict", "version": "3.12",
        "keywords": ["dict", "keys", "values", "items", "get", "update", "setdefault", "defaultdict"],
        "content": (
            "Dictionaries map unique keys to values. dict.get(key, default) returns a value "
            "or default instead of raising KeyError. dict.setdefault(key, default) sets and "
            "returns default only if key is absent. dict.update(other) merges another mapping. "
            "dict.keys(), dict.values(), dict.items() return views. "
            "Dict comprehensions: {k: v for k, v in iterable}. "
            "Since Python 3.7 dicts preserve insertion order. "
            "For counting: use collections.Counter. "
            "For missing-key defaults: use collections.defaultdict(factory)."
        ),
    },
    {
        "id": "py-str-01", "library": "python", "topic": "string", "version": "3.12",
        "keywords": ["string", "str", "format", "split", "join", "strip", "replace", "f-string", "encode"],
        "content": (
            "Strings are immutable sequences of Unicode code points. "
            "str.split(sep=None, maxsplit=-1) splits on whitespace by default. "
            "str.join(iterable) is the inverse: sep.join(['a','b']) -> 'a<sep>b'. "
            "str.strip(chars=None) removes leading/trailing whitespace or specified chars. "
            "str.replace(old, new, count=-1) returns a new string with replacements. "
            "str.format(**kwargs) or f-strings (f'Hello {name}') for interpolation. "
            "str.encode(encoding='utf-8') converts to bytes. "
            "str.startswith(prefix), str.endswith(suffix) return booleans."
        ),
    },
    {
        "id": "py-generators-01", "library": "python", "topic": "generators", "version": "3.12",
        "keywords": ["generator", "yield", "iterator", "next", "lazy", "list comprehension", "memory"],
        "content": (
            "Generators produce values lazily — they don't build the full result in memory. "
            "A generator function uses 'yield' instead of 'return'. "
            "Generator expressions: (expr for item in iterable) — like a list comprehension but lazy. "
            "Use generators when: data is large, you only need one pass, or infinite sequences. "
            "Use list comprehensions when: you need random access, len(), or multiple iterations. "
            "next(gen) fetches the next value; StopIteration is raised when exhausted. "
            "Generators are 10-100x more memory-efficient for large datasets than lists."
        ),
    },
    {
        "id": "py-dataclass-01", "library": "python", "topic": "dataclasses", "version": "3.12",
        "keywords": ["dataclass", "dataclasses", "field", "default_factory", "frozen", "asdict", "astuple"],
        "content": (
            "@dataclass auto-generates __init__, __repr__, __eq__, and optionally __hash__. "
            "field(default_factory=list) is required for mutable defaults (not `= []`). "
            "@dataclass(frozen=True) makes instances immutable and hashable. "
            "@dataclass(order=True) generates __lt__, __le__, __gt__, __ge__ from field order. "
            "dataclasses.asdict(obj) recursively converts to dict. "
            "dataclasses.fields(obj) returns metadata about each field. "
            "Post-init logic: def __post_init__(self) runs after __init__. "
            "Use frozen dataclasses as dict keys or in sets."
        ),
    },
    {
        "id": "py-async-01", "library": "python", "topic": "asyncio", "version": "3.12",
        "keywords": ["async", "await", "asyncio", "coroutine", "event loop", "gather", "task", "aiohttp"],
        "content": (
            "async def defines a coroutine function; await suspends until the awaitable completes. "
            "asyncio.run(coroutine()) starts the event loop from synchronous code. "
            "asyncio.gather(*awaitables) runs awaitables concurrently and returns all results. "
            "asyncio.create_task(coro) schedules a coroutine for concurrent execution. "
            "asyncio.sleep(seconds) yields control back to the event loop. "
            "Use async for for async iterators; async with for async context managers. "
            "aiohttp is the standard library for async HTTP requests. "
            "Do NOT mix blocking calls (time.sleep, requests.get) inside async functions."
        ),
    },
    {
        "id": "py-exception-01", "library": "python", "topic": "exceptions", "version": "3.12",
        "keywords": ["exception", "try", "except", "finally", "raise", "error handling", "custom exception"],
        "content": (
            "try/except/else/finally: except catches exceptions, else runs if no exception, "
            "finally always runs. Catch specific exceptions before broad ones. "
            "raise re-raises the current exception; raise ExceptionType(msg) raises new. "
            "Define custom exceptions by subclassing Exception. "
            "Exception groups (Python 3.11+): ExceptionGroup bundles multiple exceptions. "
            "except* (Python 3.11+) handles exception groups. "
            "Use contextlib.suppress(ExceptionType) to silently ignore specific exceptions. "
            "Avoid bare 'except:' — it catches even SystemExit and KeyboardInterrupt."
        ),
    },
    {
        "id": "py-dunder-01", "library": "python", "topic": "dunder methods", "version": "3.12",
        "keywords": ["__repr__", "__str__", "__eq__", "__hash__", "__len__", "__getitem__", "dunder", "magic method"],
        "content": (
            "__str__(self) defines the user-friendly string (used by print, str()). "
            "__repr__(self) defines the developer repr (used by repr(), REPL). "
            "Rule: __repr__ should return a string that ideally recreates the object. "
            "__eq__(self, other) defines equality (==). If you define __eq__, also define __hash__. "
            "__hash__ returns an int; objects that compare equal must have the same hash. "
            "__len__(self) enables len(obj). __getitem__(self, key) enables obj[key]. "
            "__contains__(self, item) enables 'item in obj'. "
            "__enter__ and __exit__ implement the context manager protocol (with statement)."
        ),
    },
    {
        "id": "pd-dataframe-01", "library": "pandas", "topic": "DataFrame basics", "version": "2.2",
        "keywords": ["dataframe", "pandas", "read_csv", "head", "shape", "dtypes", "info", "describe"],
        "content": (
            "pd.read_csv(path) reads a CSV file into a DataFrame. "
            "df.head(n=5) returns the first n rows. df.shape returns (rows, cols). "
            "df.dtypes returns column dtypes. df.info() prints summary. "
            "df.describe() computes summary stats for numeric columns. "
            "df['col'] selects a column (returns Series). df[['a','b']] selects multiple columns. "
            "df.loc[row_label, col_label] label-based indexing. "
            "df.iloc[row_int, col_int] integer-based indexing. "
            "df.copy() makes a deep copy to avoid SettingWithCopyWarning."
        ),
    },
    {
        "id": "pd-sort-01", "library": "pandas", "topic": "sorting", "version": "2.2",
        "keywords": ["sort", "sort_values", "sort_index", "ascending", "descending", "rank", "nlargest"],
        "content": (
            "df.sort_values(by='col') sorts ascending by default. "
            "df.sort_values(by=['col1','col2'], ascending=[True, False]) sorts by multiple columns "
            "with different directions. "
            "df.sort_index() sorts by the DataFrame index. "
            "df.nlargest(n, 'col') returns the n largest rows by column value. "
            "df.nsmallest(n, 'col') returns the n smallest rows. "
            "df['col'].rank(method='average') assigns ranks (handles ties). "
            "All sort methods return a new DataFrame unless inplace=True."
        ),
    },
    {
        "id": "pd-groupby-01", "library": "pandas", "topic": "groupby", "version": "2.2",
        "keywords": ["groupby", "agg", "aggregate", "transform", "apply", "pivot", "pivot_table"],
        "content": (
            "df.groupby('col').agg({'a': 'sum', 'b': 'mean'}) aggregates each group. "
            "df.groupby(['col1','col2']) groups by multiple columns. "
            "grouped.transform(func) returns same-shape result (e.g., z-scores within group). "
            "grouped.filter(lambda x: ...) drops groups not meeting a condition. "
            "df.pivot_table(values='v', index='r', columns='c', aggfunc='sum') is groupby + reshape. "
            "Use as_index=False to keep group keys as regular columns. "
            "named aggregations: df.groupby('g').agg(total=('val','sum'), mean_val=('val','mean'))."
        ),
    },
    {
        "id": "pd-merge-01", "library": "pandas", "topic": "merge and join", "version": "2.2",
        "keywords": ["merge", "join", "concat", "how", "inner", "outer", "left", "right", "on", "suffixes"],
        "content": (
            "pd.merge(left, right, on='key', how='inner') is SQL-style join. "
            "how: 'inner' (intersection), 'outer' (union), 'left', 'right'. "
            "pd.concat([df1, df2], axis=0) stacks rows; axis=1 stacks columns. "
            "Use suffixes=('_x','_y') to disambiguate overlapping column names. "
            "df.join(other, on='key') uses the index by default. "
            "merge with indicator=True adds a '_merge' column showing source. "
            "Many-to-many merges duplicate rows — validate with validate='one_to_one'."
        ),
    },
    {
        "id": "req-01", "library": "requests", "topic": "HTTP requests", "version": "2.31",
        "keywords": ["requests", "get", "post", "http", "response", "status_code", "json", "headers", "timeout"],
        "content": (
            "requests.get(url, params={}, headers={}, timeout=10) sends an HTTP GET. "
            "requests.post(url, json={}, data={}, files={}) sends POST. "
            "response.status_code is the HTTP status (200=OK, 404=Not Found, etc.). "
            "response.json() parses the response body as JSON. "
            "response.text gives the raw text. response.content gives raw bytes. "
            "response.raise_for_status() raises requests.HTTPError for 4xx/5xx. "
            "Always set timeout to avoid hanging. Use a Session for connection pooling. "
            "requests.exceptions.RequestException is the base exception for all requests errors."
        ),
    },
    {
        "id": "req-02", "library": "requests", "topic": "error handling", "version": "2.31",
        "keywords": ["requests", "error", "exception", "HTTPError", "ConnectionError", "Timeout", "retry"],
        "content": (
            "Catch requests.exceptions.HTTPError for 4xx/5xx (after raise_for_status). "
            "requests.exceptions.ConnectionError for network problems. "
            "requests.exceptions.Timeout if timeout is exceeded. "
            "requests.exceptions.RequestException catches all of the above. "
            "For retries: use urllib3.util.retry.Retry with a requests.adapters.HTTPAdapter. "
            "Pattern: session.mount('https://', HTTPAdapter(max_retries=Retry(total=3))). "
            "Check response.ok (True for 2xx) as a quick boolean check."
        ),
    },
    {
        "id": "py-typing-01", "library": "python", "topic": "type hints", "version": "3.12",
        "keywords": ["typing", "type hints", "Optional", "Union", "List", "Dict", "TypeVar", "Protocol", "annotate"],
        "content": (
            "Type hints improve readability and enable static analysis with mypy/pyright. "
            "From Python 3.10: use X | Y instead of Union[X, Y]; X | None instead of Optional[X]. "
            "From Python 3.9: use list[int] instead of List[int]; dict[str, int] instead of Dict. "
            "TypeVar: T = TypeVar('T') for generic functions. "
            "Protocol defines structural subtyping (duck typing). "
            "Literal['foo', 'bar'] restricts to specific values. "
            "Final[int] marks a variable as constant. "
            "TypedDict defines dicts with typed keys."
        ),
    },
    {
        "id": "py-pathlib-01", "library": "python", "topic": "pathlib", "version": "3.12",
        "keywords": ["pathlib", "path", "file", "directory", "read_text", "write_text", "glob", "mkdir"],
        "content": (
            "from pathlib import Path: Path is the modern file-system abstraction. "
            "Path('dir') / 'subdir' / 'file.txt' builds paths safely (handles OS separators). "
            "path.read_text(encoding='utf-8') and path.write_text(data, encoding='utf-8'). "
            "path.read_bytes() / path.write_bytes(data) for binary files. "
            "path.exists(), path.is_file(), path.is_dir() for checks. "
            "path.mkdir(parents=True, exist_ok=True) creates directory trees. "
            "path.glob('**/*.py') finds all Python files recursively. "
            "path.stem (filename without ext), path.suffix, path.parent, path.name."
        ),
    },
]

print(f"Documentation fragments loaded: {len(DOC_FRAGMENTS)}")
for doc in DOC_FRAGMENTS[:3]:
    print(f"  [{doc['id']}] {doc['library']}.{doc['topic']} — {len(doc['keywords'])} keywords")

In [ ]:
# ── Code snippets ─────────────────────────────────────────────────────────────
SNIPPETS = [
    {
        "id": "snip-list-sort", "topic": "sorting",
        "tags": ["list", "sort", "key", "lambda", "reverse"],
        "title": "Sort a list of dicts by a key",
        "code": (
            "items = [{'name': 'Alice', 'age': 30}, {'name': 'Bob', 'age': 25}]\n"
            "# Sort ascending by age\n"
            "sorted_items = sorted(items, key=lambda x: x['age'])\n"
            "# Sort descending by name\n"
            "items.sort(key=lambda x: x['name'], reverse=True)"
        ),
    },
    {
        "id": "snip-df-sort", "topic": "pandas sorting",
        "tags": ["pandas", "dataframe", "sort_values", "multi-column", "descending"],
        "title": "Sort a DataFrame by multiple columns",
        "code": (
            "import pandas as pd\n"
            "df = pd.DataFrame({'dept': ['A','B','A'], 'salary': [70000, 90000, 80000], 'years': [3, 5, 2]})\n"
            "# Sort by dept ascending, then salary descending\n"
            "result = df.sort_values(by=['dept', 'salary'], ascending=[True, False])\n"
            "print(result)"
        ),
    },
    {
        "id": "snip-generator", "topic": "generators",
        "tags": ["generator", "yield", "lazy", "memory efficient", "fibonacci"],
        "title": "Lazy generator vs list comprehension",
        "code": (
            "# Generator expression — memory efficient, no list built\n"
            "gen = (x ** 2 for x in range(1_000_000))\n"
            "first_10 = [next(gen) for _ in range(10)]\n\n"
            "# Generator function\n"
            "def fibonacci():\n"
            "    a, b = 0, 1\n"
            "    while True:\n"
            "        yield a\n"
            "        a, b = b, a + b\n\n"
            "fib = fibonacci()\n"
            "print([next(fib) for _ in range(10)])"
        ),
    },
    {
        "id": "snip-dataclass", "topic": "dataclasses",
        "tags": ["dataclass", "field", "default_factory", "frozen", "post_init"],
        "title": "Dataclass with default_factory and post-init validation",
        "code": (
            "from dataclasses import dataclass, field\n\n"
            "@dataclass\n"
            "class Task:\n"
            "    title: str\n"
            "    priority: int = 1\n"
            "    tags: list[str] = field(default_factory=list)  # NOT tags: list = []\n"
            "    _id: str = field(init=False, repr=False)\n\n"
            "    def __post_init__(self):\n"
            "        if not (1 <= self.priority <= 5):\n"
            "            raise ValueError(f'Priority must be 1-5, got {self.priority}')\n"
            "        import uuid\n"
            "        self._id = str(uuid.uuid4())[:8]\n\n"
            "t = Task('Write docs', priority=2, tags=['python', 'docs'])\n"
            "print(t)"
        ),
    },
    {
        "id": "snip-async", "topic": "asyncio",
        "tags": ["async", "await", "gather", "asyncio", "concurrent"],
        "title": "async/await with concurrent tasks using asyncio.gather",
        "code": (
            "import asyncio\n\n"
            "async def fetch(url: str, delay: float) -> str:\n"
            "    await asyncio.sleep(delay)  # simulate network I/O\n"
            "    return f'Response from {url}'\n\n"
            "async def main():\n"
            "    urls = ['https://api.example.com/a', 'https://api.example.com/b']\n"
            "    # Run both fetches concurrently (not sequentially)\n"
            "    results = await asyncio.gather(*[fetch(u, 0.1) for u in urls])\n"
            "    for r in results:\n"
            "        print(r)\n\n"
            "asyncio.run(main())"
        ),
    },
    {
        "id": "snip-requests-error", "topic": "requests error handling",
        "tags": ["requests", "HTTPError", "exception", "retry", "raise_for_status"],
        "title": "Robust HTTP request with error handling and retry",
        "code": (
            "import requests\n"
            "from requests.adapters import HTTPAdapter\n"
            "from urllib3.util.retry import Retry\n\n"
            "session = requests.Session()\n"
            "retry = Retry(total=3, backoff_factor=0.5, status_forcelist=[500, 502, 503])\n"
            "session.mount('https://', HTTPAdapter(max_retries=retry))\n\n"
            "try:\n"
            "    resp = session.get('https://api.example.com/data', timeout=10)\n"
            "    resp.raise_for_status()  # raises HTTPError for 4xx/5xx\n"
            "    data = resp.json()\n"
            "except requests.exceptions.HTTPError as e:\n"
            "    print(f'HTTP error: {e.response.status_code}')\n"
            "except requests.exceptions.Timeout:\n"
            "    print('Request timed out')\n"
            "except requests.exceptions.ConnectionError:\n"
            "    print('Network error')"
        ),
    },
    {
        "id": "snip-repr-str", "topic": "dunder methods",
        "tags": ["__repr__", "__str__", "dunder", "class", "magic method"],
        "title": "__repr__ vs __str__ — when to use each",
        "code": (
            "class Point:\n"
            "    def __init__(self, x: float, y: float):\n"
            "        self.x = x\n"
            "        self.y = y\n\n"
            "    def __repr__(self) -> str:\n"
            "        # Should ideally reconstruct the object\n"
            "        return f'Point(x={self.x!r}, y={self.y!r})'\n\n"
            "    def __str__(self) -> str:\n"
            "        # Human-friendly: used by print() and str()\n"
            "        return f'({self.x}, {self.y})'\n\n"
            "p = Point(3.0, 4.0)\n"
            "print(repr(p))  # Point(x=3.0, y=4.0)  — for developers\n"
            "print(str(p))   # (3.0, 4.0)            — for users\n"
            "print(p)        # (3.0, 4.0)            — print() uses __str__"
        ),
    },
    {
        "id": "snip-pathlib", "topic": "pathlib",
        "tags": ["pathlib", "Path", "file", "read", "write", "glob"],
        "title": "Read, write, and search files with pathlib",
        "code": (
            "from pathlib import Path\n\n"
            "# Build a path safely (works on Windows and Linux)\n"
            "data_dir = Path('data') / 'processed'\n"
            "data_dir.mkdir(parents=True, exist_ok=True)\n\n"
            "# Write and read text\n"
            "file = data_dir / 'output.txt'\n"
            "file.write_text('Hello, world!', encoding='utf-8')\n"
            "content = file.read_text(encoding='utf-8')\n\n"
            "# Find all Python files recursively\n"
            "py_files = list(Path('.').glob('**/*.py'))\n"
            "print(f'Found {len(py_files)} Python files')"
        ),
    },
    {
        "id": "snip-list-comp", "topic": "list comprehension",
        "tags": ["list comprehension", "filter", "map", "conditional", "nested"],
        "title": "List comprehension patterns (filter, map, nested)",
        "code": (
            "numbers = list(range(1, 11))\n\n"
            "# Filter: keep evens\n"
            "evens = [x for x in numbers if x % 2 == 0]\n\n"
            "# Map: square all\n"
            "squares = [x ** 2 for x in numbers]\n\n"
            "# Filter + map: square only odd numbers\n"
            "odd_squares = [x**2 for x in numbers if x % 2 != 0]\n\n"
            "# Nested: flatten a 2D list\n"
            "matrix = [[1,2,3],[4,5,6],[7,8,9]]\n"
            "flat = [cell for row in matrix for cell in row]\n\n"
            "# Dict comprehension\n"
            "word_lengths = {w: len(w) for w in ['hello', 'world', 'python']}\n"
            "print(evens, squares[:3], flat)"
        ),
    },
    {
        "id": "snip-defaultdict", "topic": "defaultdict",
        "tags": ["defaultdict", "collections", "dict", "grouping", "counter"],
        "title": "Grouping items with defaultdict",
        "code": (
            "from collections import defaultdict, Counter\n\n"
            "# Group strings by first letter\n"
            "words = ['apple', 'avocado', 'banana', 'blueberry', 'cherry']\n"
            "by_letter = defaultdict(list)\n"
            "for word in words:\n"
            "    by_letter[word[0]].append(word)\n"
            "print(dict(by_letter))  # {'a': ['apple','avocado'], 'b': [...], ...}\n\n"
            "# Count occurrences\n"
            "text = 'the quick brown fox jumps over the lazy dog'\n"
            "word_count = Counter(text.split())\n"
            "print(word_count.most_common(3))"
        ),
    },
]

print(f"Code snippets loaded: {len(SNIPPETS)}")
for s in SNIPPETS[:3]:
    print(f"  [{s['id']}] {s['title']}")

In [ ]:
# ── Simulated repository files ────────────────────────────────────────────────
REPO_FILES = {
    "src/models/user.py": (
        "from dataclasses import dataclass, field\n"
        "from typing import Optional\n\n"
        "@dataclass\n"
        "class User:\n"
        "    id: int\n"
        "    name: str\n"
        "    email: str\n"
        "    roles: list[str] = field(default_factory=list)\n"
        "    active: bool = True\n\n"
        "    def is_admin(self) -> bool:\n"
        "        return 'admin' in self.roles\n"
    ),
    "src/utils/http.py": (
        "import requests\n"
        "from requests.adapters import HTTPAdapter\n"
        "from urllib3.util.retry import Retry\n\n"
        "def make_session(retries: int = 3, timeout: int = 10) -> requests.Session:\n"
        "    session = requests.Session()\n"
        "    retry = Retry(total=retries, backoff_factor=0.5)\n"
        "    session.mount('https://', HTTPAdapter(max_retries=retry))\n"
        "    return session\n"
    ),
    "src/data/pipeline.py": (
        "import pandas as pd\n"
        "from pathlib import Path\n\n"
        "def load_and_clean(path: Path) -> pd.DataFrame:\n"
        "    df = pd.read_csv(path)\n"
        "    df = df.dropna()\n"
        "    df.columns = [c.strip().lower().replace(' ', '_') for c in df.columns]\n"
        "    return df\n\n"
        "def sort_and_rank(df: pd.DataFrame, col: str) -> pd.DataFrame:\n"
        "    return df.sort_values(col, ascending=False).reset_index(drop=True)\n"
    ),
    "tests/test_user.py": (
        "import pytest\n"
        "from src.models.user import User\n\n"
        "def test_user_is_admin():\n"
        "    u = User(1, 'Alice', 'alice@test.com', roles=['admin'])\n"
        "    assert u.is_admin()\n\n"
        "def test_user_not_admin():\n"
        "    u = User(2, 'Bob', 'bob@test.com')\n"
        "    assert not u.is_admin()\n"
    ),
    "src/async_worker.py": (
        "import asyncio\n"
        "from typing import Callable, Awaitable\n\n"
        "async def run_tasks(tasks: list[Callable[[], Awaitable]]) -> list:\n"
        "    return await asyncio.gather(*[t() for t in tasks])\n\n"
        "async def with_timeout(coro, seconds: float):\n"
        "    return await asyncio.wait_for(coro, timeout=seconds)\n"
    ),
}

# ── Function signatures ────────────────────────────────────────────────────────
FUNCTION_SIGNATURES = {
    "sorted": {
        "signature": "sorted(iterable, /, *, key=None, reverse=False)",
        "returns": "list",
        "module": "builtins",
        "docstring": "Return a new sorted list from the items in iterable. key specifies a function of one argument that is used to extract a comparison key. reverse is a boolean; if set to True, the sorted list is returned reversed.",
    },
    "pd.DataFrame.sort_values": {
        "signature": "DataFrame.sort_values(by, *, axis=0, ascending=True, inplace=False, kind='quicksort', na_position='last', ignore_index=False, key=None)",
        "returns": "DataFrame or None",
        "module": "pandas",
        "docstring": "Sort by the values along either axis. by: str or list of str — name or list of names to sort by. ascending: bool or list of bool. na_position: 'first' or 'last' puts NaNs at the beginning or end.",
    },
    "asyncio.gather": {
        "signature": "asyncio.gather(*aws, return_exceptions=False)",
        "returns": "list of results",
        "module": "asyncio",
        "docstring": "Run awaitable objects in aws concurrently. If return_exceptions=False (default), the first raised exception is immediately propagated. If True, exceptions are aggregated in the result list.",
    },
    "dataclasses.field": {
        "signature": "dataclasses.field(*, default=MISSING, default_factory=MISSING, init=True, repr=True, hash=None, compare=True, metadata=None, kw_only=MISSING)",
        "returns": "Field",
        "module": "dataclasses",
        "docstring": "Returns a Field object for use with @dataclass. Use default_factory for mutable defaults (list, dict). init=False fields are not included in __init__. repr=False fields are not shown in __repr__.",
    },
    "requests.Session.get": {
        "signature": "Session.get(url, **kwargs)",
        "returns": "requests.Response",
        "module": "requests",
        "docstring": "Sends a GET request. kwargs may include: params (dict/bytes), headers (dict), timeout (float/tuple), allow_redirects (bool), auth (tuple/AuthBase), verify (bool/str), cert (str/tuple).",
    },
    "Path.glob": {
        "signature": "Path.glob(pattern)",
        "returns": "Generator[Path, None, None]",
        "module": "pathlib",
        "docstring": "Glob the given relative pattern in the directory represented by this path, yielding Path objects matching the pattern. '**' matches any number of directories.",
    },
}

print(f"Repo files: {len(REPO_FILES)}")
print(f"Function signatures: {len(FUNCTION_SIGNATURES)}")

## 3. Data Model — Tool Results & Answers

In [ ]:
class QueryIntent(Enum):
    LOOKUP = "lookup"         # "What is X?" / "What does X do?"
    EXPLAIN = "explain"       # "How does X work?" / "Why use X?"
    GENERATE = "generate"     # "Show me an example of X" / "How to do X"
    DEBUG = "debug"           # "Why does X fail?" / "Fix this error"
    COMPARE = "compare"       # "What is the difference between X and Y"

    def __str__(self):
        return self.value


@dataclass
class ContextPiece:
    """A single retrieved context item."""
    source_id: str
    source_type: str        # "doc", "snippet", "signature", "repo"
    title: str
    content: str
    score: float            # Relevance 0.0–1.0
    library: str = ""

    def __repr__(self):
        return f"ContextPiece({self.source_type}:{self.source_id!r}, score={self.score:.2f})"


@dataclass
class ToolCall:
    """A record of a single tool invocation."""
    tool_name: str
    query: str
    results: list[ContextPiece]
    latency_ms: float
    called_at: str = ""

    def __post_init__(self):
        if not self.called_at:
            self.called_at = datetime.now().isoformat()


@dataclass
class CodingAnswer:
    """The structured answer produced by the assistant."""
    question: str
    intent: QueryIntent
    explanation: str
    code_block: str         # Primary code example
    key_points: list[str]
    citations: list[str]    # Source IDs
    tool_calls: list[ToolCall]
    follow_up_suggestions: list[str]
    answered_at: str = ""

    def __post_init__(self):
        if not self.answered_at:
            self.answered_at = datetime.now().isoformat()

    def display(self):
        width = 72
        print(f"\n{'='*width}")
        print(f" CODING ASSISTANT")
        print(f"{'='*width}")
        print(f" Q: {self.question}")
        print(f" Intent: {self.intent}  |  Tools called: {len(self.tool_calls)}")
        print(f"{'─'*width}")
        print(textwrap.fill(self.explanation, width=width - 2, initial_indent=" ", subsequent_indent=" "))
        if self.code_block:
            print(f"\n {'─'*20} Code {'─'*20}")
            for line in self.code_block.split("\n"):
                print(f"  {line}")
        if self.key_points:
            print(f"\n Key Points:")
            for pt in self.key_points:
                print(f"   • {pt}")
        if self.citations:
            print(f"\n Sources: {', '.join(self.citations)}")
        if self.follow_up_suggestions:
            print(f"\n Related questions:")
            for q in self.follow_up_suggestions:
                print(f"   → {q}")
        print(f"{'='*width}\n")


print("Data model loaded.")

## 4. Tool Implementations

Each tool is a pure function: it takes a query and returns a list of `ContextPiece` objects with relevance scores.

### Scoring Strategy

`search_docs` and `search_snippets` use **TF-IDF-style keyword matching** — the score is the fraction of query tokens that match the item's keywords, boosted by exact phrase matches.


In [ ]:
import time as _time


def _tokenize(text: str) -> set[str]:
    """Lowercase, split on non-alphanumeric, remove short tokens."""
    tokens = re.findall(r"[a-z][a-z0-9_]*", text.lower())
    return {t for t in tokens if len(t) > 1}


def _keyword_score(query_tokens: set[str], keywords: list[str], content: str) -> float:
    """Return a relevance score in [0, 1] based on token overlap."""
    kw_tokens = set()
    for kw in keywords:
        kw_tokens.update(_tokenize(kw))
    content_tokens = _tokenize(content)
    all_tokens = kw_tokens | content_tokens

    if not all_tokens:
        return 0.0

    keyword_hits = len(query_tokens & kw_tokens) / max(len(kw_tokens), 1)
    content_hits = len(query_tokens & content_tokens) / max(len(query_tokens), 1)
    return round(min(0.6 * keyword_hits + 0.4 * content_hits, 1.0), 3)


def search_docs(query: str, top_k: int = 3) -> ToolCall:
    """Search documentation fragments by keyword similarity."""
    t0 = _time.perf_counter()
    tokens = _tokenize(query)

    results = []
    for doc in DOC_FRAGMENTS:
        score = _keyword_score(tokens, doc["keywords"], doc["content"])
        if score > 0.05:
            results.append(ContextPiece(
                source_id=doc["id"],
                source_type="doc",
                title=f"{doc['library']}.{doc['topic']}",
                content=doc["content"],
                score=score,
                library=doc["library"],
            ))

    results.sort(key=lambda x: x.score, reverse=True)
    latency = (_time.perf_counter() - t0) * 1000

    return ToolCall(
        tool_name="search_docs",
        query=query,
        results=results[:top_k],
        latency_ms=round(latency, 2),
    )


def search_snippets(query: str, top_k: int = 3) -> ToolCall:
    """Search code snippets by tag and title similarity."""
    t0 = _time.perf_counter()
    tokens = _tokenize(query)

    results = []
    for snip in SNIPPETS:
        all_text = snip["title"] + " " + " ".join(snip["tags"]) + " " + snip["code"]
        score = _keyword_score(tokens, snip["tags"], all_text)
        if score > 0.05:
            results.append(ContextPiece(
                source_id=snip["id"],
                source_type="snippet",
                title=snip["title"],
                content=snip["code"],
                score=score,
                library=snip["tags"][0] if snip["tags"] else "",
            ))

    results.sort(key=lambda x: x.score, reverse=True)
    latency = (_time.perf_counter() - t0) * 1000

    return ToolCall(
        tool_name="search_snippets",
        query=query,
        results=results[:top_k],
        latency_ms=round(latency, 2),
    )


def lookup_function(name: str) -> ToolCall:
    """Exact lookup of a function signature and docstring."""
    t0 = _time.perf_counter()
    # Try exact match first, then partial
    match = FUNCTION_SIGNATURES.get(name)
    if not match:
        name_lower = name.lower()
        for key, val in FUNCTION_SIGNATURES.items():
            if name_lower in key.lower():
                match = val
                name = key
                break

    results = []
    if match:
        results.append(ContextPiece(
            source_id=f"sig-{name.replace('.','_')}",
            source_type="signature",
            title=f"{name} signature",
            content=f"Signature: {match['signature']}\nReturns: {match['returns']}\n\n{match['docstring']}",
            score=1.0,
            library=match["module"],
        ))

    latency = (_time.perf_counter() - t0) * 1000
    return ToolCall(
        tool_name="lookup_function",
        query=name,
        results=results,
        latency_ms=round(latency, 2),
    )


def get_examples(topic: str, top_k: int = 2) -> ToolCall:
    """Find runnable code examples for a topic."""
    t0 = _time.perf_counter()
    tokens = _tokenize(topic)

    results = []
    for snip in SNIPPETS:
        tag_score = len(tokens & set(_tokenize(" ".join(snip["tags"])))) / max(len(tokens), 1)
        topic_score = len(tokens & _tokenize(snip["topic"])) / max(len(tokens), 1)
        score = max(tag_score, topic_score)
        if score > 0.1:
            results.append(ContextPiece(
                source_id=snip["id"],
                source_type="snippet",
                title=snip["title"],
                content=snip["code"],
                score=round(score, 3),
                library=snip.get("tags", [""])[0],
            ))

    results.sort(key=lambda x: x.score, reverse=True)
    latency = (_time.perf_counter() - t0) * 1000
    return ToolCall(
        tool_name="get_examples",
        query=topic,
        results=results[:top_k],
        latency_ms=round(latency, 2),
    )


def search_repo(pattern: str, top_k: int = 3) -> ToolCall:
    """Search simulated repository files by content pattern."""
    t0 = _time.perf_counter()
    tokens = _tokenize(pattern)

    results = []
    for filepath, content in REPO_FILES.items():
        score = _keyword_score(tokens, [filepath], content)
        if score > 0.05:
            results.append(ContextPiece(
                source_id=filepath,
                source_type="repo",
                title=filepath,
                content=content,
                score=score,
                library="repo",
            ))

    results.sort(key=lambda x: x.score, reverse=True)
    latency = (_time.perf_counter() - t0) * 1000
    return ToolCall(
        tool_name="search_repo",
        query=pattern,
        results=results[:top_k],
        latency_ms=round(latency, 2),
    )


def run_code(snippet: str) -> ToolCall:
    """Execute a Python snippet and return stdout/result as context."""
    t0 = _time.perf_counter()
    
    # Safety: only allow snippets that don't import dangerous modules
    BLOCKED = {"os", "subprocess", "socket", "shutil", "ctypes"}
    imports = re.findall(r"import\s+(\w+)", snippet)
    for imp in imports:
        if imp in BLOCKED:
            latency = (_time.perf_counter() - t0) * 1000
            return ToolCall(
                tool_name="run_code",
                query=snippet[:50],
                results=[ContextPiece(
                    source_id="run-blocked",
                    source_type="execution",
                    title="Blocked import",
                    content=f"Blocked: import '{imp}' is not allowed in sandbox.",
                    score=0.0,
                )],
                latency_ms=round(latency, 2),
            )

    # Capture stdout
    import sys
    old_stdout = sys.stdout
    sys.stdout = buf = StringIO()
    output = ""
    try:
        exec(compile(snippet, "<sandbox>", "exec"), {})  # noqa: S102
        output = buf.getvalue().strip()
    except Exception as e:
        output = f"Error: {type(e).__name__}: {e}"
    finally:
        sys.stdout = old_stdout

    latency = (_time.perf_counter() - t0) * 1000
    return ToolCall(
        tool_name="run_code",
        query=snippet[:60],
        results=[ContextPiece(
            source_id="run-output",
            source_type="execution",
            title="Code execution output",
            content=output or "(no output)",
            score=1.0,
        )],
        latency_ms=round(latency, 2),
    )


# Quick smoke tests
r1 = search_docs("how to sort a pandas dataframe")
r2 = search_snippets("async await gather")
r3 = lookup_function("asyncio.gather")
print(f"search_docs     → {len(r1.results)} results, top: {r1.results[0].source_id if r1.results else 'none'}")
print(f"search_snippets → {len(r2.results)} results, top: {r2.results[0].source_id if r2.results else 'none'}")
print(f"lookup_function → {len(r3.results)} results, top: {r3.results[0].source_id if r3.results else 'none'}")

## 5. Intent Classifier & Tool Router

The **intent classifier** maps a natural-language question to a `QueryIntent` enum value using keyword pattern matching.

The **tool router** then decides which tools to call based on the intent:

| Intent | Tools called |
|---|---|
| `LOOKUP` | `lookup_function`, `search_docs` |
| `EXPLAIN` | `search_docs`, `search_snippets` |
| `GENERATE` | `get_examples`, `search_snippets` |
| `DEBUG` | `search_docs`, `search_snippets`, `run_code` |
| `COMPARE` | `search_docs` (×2), `search_snippets` |


In [ ]:
# Intent classification rules — checked in order; first match wins
INTENT_RULES = [
    (QueryIntent.COMPARE,  [r"\bdiff(?:erence)?\b", r"\bvs\b", r"\bversus\b", r"\bcompare\b", r"\bwhen to use\b"]),
    (QueryIntent.DEBUG,    [r"\berror\b", r"\bfail\b", r"\bfix\b", r"\bbug\b", r"\bwhy (?:does|is|do)\b", r"\bnot work\b"]),
    (QueryIntent.GENERATE, [r"\bshow\b.*\bexample\b", r"\bhow (?:do|to|can)\b", r"\bexample\b", r"\bsample\b", r"\bwrite\b.*\bcode\b"]),
    (QueryIntent.EXPLAIN,  [r"\bhow does\b", r"\bexplain\b", r"\bwhat is\b.*\bmean\b", r"\bwhy\b"]),
    (QueryIntent.LOOKUP,   [r"\bwhat is\b", r"\bwhat does\b", r"\bsignature\b", r"\bsyntax\b", r"\bdocstring\b"]),
]


def classify_intent(question: str) -> QueryIntent:
    """Classify the question's intent using regex heuristics."""
    q = question.lower()
    for intent, patterns in INTENT_RULES:
        for pattern in patterns:
            if re.search(pattern, q):
                return intent
    return QueryIntent.EXPLAIN  # Default


def extract_keywords(question: str) -> list[str]:
    """Extract likely subject keywords from the question."""
    # Strip question words and return content words
    STOP = {"what", "how", "why", "when", "where", "which", "who", "is", "are",
            "do", "does", "can", "the", "a", "an", "to", "in", "of", "for",
            "and", "or", "with", "use", "using", "vs", "versus", "difference",
            "between", "show", "me", "example", "code", "write", "explain"}
    q = question.lower()
    q = re.sub(r"[^\w\s]", " ", q)
    tokens = q.split()
    return [t for t in tokens if t not in STOP and len(t) > 2]


def route_tools(question: str, intent: QueryIntent) -> list[str]:
    """Decide which tools to call based on intent."""
    keywords = extract_keywords(question)
    q_lower = question.lower()

    tools_to_call = []

    if intent == QueryIntent.LOOKUP:
        tools_to_call = ["lookup_function", "search_docs"]
    elif intent == QueryIntent.EXPLAIN:
        tools_to_call = ["search_docs", "search_snippets"]
    elif intent == QueryIntent.GENERATE:
        tools_to_call = ["get_examples", "search_snippets", "search_docs"]
    elif intent == QueryIntent.DEBUG:
        tools_to_call = ["search_docs", "search_snippets"]
    elif intent == QueryIntent.COMPARE:
        tools_to_call = ["search_docs", "search_snippets"]

    # Always add search_repo if code-level detail is expected
    if any(kw in q_lower for kw in ["class", "function", "module", "import", "file"]):
        if "search_repo" not in tools_to_call:
            tools_to_call.append("search_repo")

    return tools_to_call


# Demo
for q in [
    "How do I sort a DataFrame by multiple columns?",
    "What is the difference between __repr__ and __str__?",
    "Show me async/await example",
    "Why does my requests call fail with timeout?",
    "What does asyncio.gather do?",
]:
    intent = classify_intent(q)
    kws = extract_keywords(q)
    tools = route_tools(q, intent)
    print(f"Q: {q[:55]:<55} intent={str(intent):<10} tools={tools}")

## 6. Context Builder

The context builder:
1. Calls each routed tool with the user's question
2. Collects all `ContextPiece` objects into a pool
3. Deduplicates by `source_id`
4. Re-ranks by combined score
5. Trims to a maximum context budget


In [ ]:
MAX_CONTEXT_PIECES = 6

TOOL_FN_MAP = {
    "search_docs": search_docs,
    "search_snippets": search_snippets,
    "lookup_function": lookup_function,
    "get_examples": get_examples,
    "search_repo": search_repo,
    "run_code": run_code,
}


def build_context(question: str, tool_names: list[str],
                  keywords: list[str]) -> tuple[list[ContextPiece], list[ToolCall]]:
    """Call tools and build a deduplicated, ranked context window."""
    all_calls: list[ToolCall] = []
    seen_ids: set[str] = set()
    pool: list[ContextPiece] = []

    for tool_name in tool_names:
        fn = TOOL_FN_MAP.get(tool_name)
        if not fn:
            continue

        # For lookup_function, try each keyword as the function name
        if tool_name == "lookup_function":
            # Try to find a plausible function name in the question
            func_candidates = []
            q_lower = question.lower()
            for kw in keywords:
                if "." in q_lower:
                    # Grab dotted names from question
                    dotted = re.findall(r"[\w]+\.[\w.]+", q_lower)
                    func_candidates.extend(dotted)
                func_candidates.append(kw)
            call = fn(question)  # Fallback: pass the whole question
            if not call.results:
                for cand in func_candidates[:3]:
                    call = fn(cand)
                    if call.results:
                        break
        else:
            call = fn(question)

        all_calls.append(call)

        for piece in call.results:
            if piece.source_id not in seen_ids:
                seen_ids.add(piece.source_id)
                pool.append(piece)

    # Re-rank: doc and signature pieces get a type boost, snippets especially valued for GENERATE
    for piece in pool:
        if piece.source_type == "signature":
            piece.score = min(piece.score * 1.3, 1.0)
        elif piece.source_type == "snippet":
            piece.score = min(piece.score * 1.15, 1.0)

    pool.sort(key=lambda x: x.score, reverse=True)

    return pool[:MAX_CONTEXT_PIECES], all_calls


# Demo
ctx_pieces, ctx_calls = build_context(
    "How do I sort a pandas DataFrame by multiple columns?",
    ["search_docs", "search_snippets", "lookup_function"],
    ["sort", "dataframe", "pandas", "columns"],
)
print(f"Context pieces: {len(ctx_pieces)}")
for p in ctx_pieces:
    print(f"  [{p.source_type:9s}] {p.source_id:<25} score={p.score:.3f}  {p.title[:40]}")

## 7. Answer Synthesizer

The synthesizer combines all context pieces into a structured `CodingAnswer`. In production this is where you'd call an LLM (GPT-4o, Claude, Gemini, etc.) with the context as the prompt. Here we use a **template engine** to show the data flow and structure.

The synthesizer:
1. Picks the best snippet from context → `code_block`
2. Extracts key facts from doc pieces → `key_points`
3. Writes an explanation paragraph from the top doc piece
4. Suggests follow-up questions based on related topics


In [ ]:
FOLLOW_UPS: dict[str, list[str]] = {
    "pandas":    ["How do I group a DataFrame and compute aggregates?",
                  "How do I merge two DataFrames on a key column?",
                  "How do I filter rows with multiple conditions in pandas?"],
    "asyncio":   ["How does asyncio.gather differ from asyncio.wait?",
                  "How do I handle exceptions in async tasks?",
                  "When should I use asyncio.create_task vs await?"],
    "requests":  ["How do I send POST requests with JSON data?",
                  "How do I add authentication headers to requests?",
                  "What is the difference between requests.get and requests.Session.get?"],
    "dataclass": ["How do I make a dataclass frozen (immutable)?",
                  "How do dataclasses compare to NamedTuple?",
                  "How do I serialize a dataclass to JSON?"],
    "list":      ["What is the difference between list.sort() and sorted()?",
                  "How do I flatten a nested list?",
                  "When should I use a tuple instead of a list?"],
    "generator": ["How do I chain multiple generators together?",
                  "What is itertools.islice used for?",
                  "How do I convert a generator to a list?"],
}


def _extract_key_points(context: list[ContextPiece]) -> list[str]:
    """Extract bullet-point facts from doc context pieces."""
    points = []
    for piece in context:
        if piece.source_type not in ("doc", "signature"):
            continue
        # Split content on sentence boundaries and take key ones
        sentences = re.split(r"(?<=[.!?])\s+", piece.content)
        for sent in sentences[:3]:
            sent = sent.strip()
            if len(sent) > 30 and len(points) < 5:
                # Clean up and shorten
                if len(sent) > 120:
                    sent = sent[:117] + "..."
                points.append(sent)
        if len(points) >= 5:
            break
    return points


def _pick_code_block(context: list[ContextPiece]) -> str:
    """Select the best code snippet from context."""
    for piece in sorted(context, key=lambda x: x.score, reverse=True):
        if piece.source_type in ("snippet", "repo", "execution"):
            return piece.content
    return ""


def _suggest_follow_ups(context: list[ContextPiece]) -> list[str]:
    """Suggest related questions based on context libraries."""
    libs_seen = {p.library.lower() for p in context if p.library}
    suggestions = []
    for lib in libs_seen:
        for key, qs in FOLLOW_UPS.items():
            if key in lib:
                suggestions.extend(qs)
    # Deduplicate and limit
    seen = set()
    result = []
    for q in suggestions:
        if q not in seen:
            seen.add(q)
            result.append(q)
        if len(result) >= 3:
            break
    return result


def synthesize_answer(question: str, intent: QueryIntent,
                      context: list[ContextPiece],
                      tool_calls: list[ToolCall]) -> CodingAnswer:
    """Build a structured answer from context pieces."""

    # Pick primary explanation from top doc
    top_doc = next((p for p in context if p.source_type in ("doc", "signature")), None)
    if top_doc:
        # Trim to first 2 sentences for the explanation paragraph
        sentences = re.split(r"(?<=[.!?])\s+", top_doc.content)
        explanation = " ".join(sentences[:2]).strip()
        explanation = re.sub(r"\s+", " ", explanation)
    else:
        explanation = "Here is what I found based on your question:"

    # Prefix explanation based on intent
    prefixes = {
        QueryIntent.LOOKUP:   "According to the documentation: ",
        QueryIntent.EXPLAIN:  "Let me explain: ",
        QueryIntent.GENERATE: "Here is how to do it: ",
        QueryIntent.DEBUG:    "The likely cause: ",
        QueryIntent.COMPARE:  "Key differences: ",
    }
    full_explanation = prefixes.get(intent, "") + explanation

    code_block = _pick_code_block(context)
    key_points = _extract_key_points(context)
    citations = list(dict.fromkeys(p.source_id for p in context))
    follow_ups = _suggest_follow_ups(context)

    return CodingAnswer(
        question=question,
        intent=intent,
        explanation=full_explanation,
        code_block=code_block,
        key_points=key_points,
        citations=citations,
        tool_calls=tool_calls,
        follow_up_suggestions=follow_ups,
    )


print("Synthesizer loaded.")

## 8. CodingAssistant — Full Agent

The `CodingAssistant` class wires together all five components into a single `ask()` method:

```text
ask(question)
  → classify_intent(question)    → intent
  → extract_keywords(question)   → keywords
  → route_tools(question, intent)→ tool list
  → build_context(...)           → [ContextPiece], [ToolCall]
  → synthesize_answer(...)       → CodingAnswer
```


In [ ]:
class CodingAssistant:
    """Tool-assisted coding Q&A agent."""

    def __init__(self):
        self._history: list[CodingAnswer] = []

    def ask(self, question: str, verbose: bool = False) -> CodingAnswer:
        """Answer a coding question using tool-assisted context retrieval."""

        # Step 1: Understand the query
        intent = classify_intent(question)
        keywords = extract_keywords(question)
        tool_names = route_tools(question, intent)

        if verbose:
            print(f"[Intent]   {intent}")
            print(f"[Keywords] {keywords}")
            print(f"[Tools]    {tool_names}")

        # Step 2: Retrieve context
        context, tool_calls = build_context(question, tool_names, keywords)

        if verbose:
            print(f"[Context]  {len(context)} pieces retrieved:")
            for p in context:
                print(f"           {p.source_type:9s} | score={p.score:.2f} | {p.title[:45]}")

        # Step 3: Synthesize answer
        answer = synthesize_answer(question, intent, context, tool_calls)
        self._history.append(answer)
        return answer

    @property
    def history(self) -> list[CodingAnswer]:
        return self._history

    def stats(self) -> dict:
        if not self._history:
            return {}
        total_tools = sum(len(a.tool_calls) for a in self._history)
        total_latency = sum(tc.latency_ms for a in self._history for tc in a.tool_calls)
        return {
            "questions_answered": len(self._history),
            "total_tool_calls": total_tools,
            "avg_tools_per_question": round(total_tools / len(self._history), 1),
            "total_latency_ms": round(total_latency, 1),
            "intent_distribution": {
                str(intent): sum(1 for a in self._history if a.intent == intent)
                for intent in QueryIntent
            },
        }


assistant = CodingAssistant()
print("CodingAssistant ready.")

## 9. Demo — Live Q&A Sessions

In [ ]:
# ── Question 1: Sort DataFrame ────────────────────────────────────────────────
q1 = "How do I sort a pandas DataFrame by multiple columns?"
a1 = assistant.ask(q1, verbose=True)
print()
a1.display()

In [ ]:
# ── Question 2: __repr__ vs __str__ ──────────────────────────────────────────
q2 = "What is the difference between __repr__ and __str__ in Python?"
a2 = assistant.ask(q2, verbose=True)
print()
a2.display()

In [ ]:
# ── Question 3: async/await example ──────────────────────────────────────────
q3 = "Show me an example of async/await with asyncio.gather"
a3 = assistant.ask(q3, verbose=True)
print()
a3.display()

In [ ]:
# ── Question 4: requests error handling ──────────────────────────────────────
q4 = "How do I handle HTTP errors in the requests library?"
a4 = assistant.ask(q4, verbose=True)
print()
a4.display()

In [ ]:
# ── Question 5: dataclasses default_factory ──────────────────────────────────
q5 = "How to use dataclasses with default_factory for mutable fields?"
a5 = assistant.ask(q5, verbose=True)
print()
a5.display()

In [ ]:
# ── Question 6: list comprehension vs generator ───────────────────────────────
q6 = "What is the difference between list comprehension and generator expression?"
a6 = assistant.ask(q6, verbose=True)
print()
a6.display()

## 10. Live Code Execution Tool Demo

The `run_code` tool can execute a code snippet in a sandboxed environment and return its output as context.


In [ ]:
# Test run_code tool directly on the DataFrame sort snippet
snip_code = SNIPPETS[1]["code"]   # Sort a DataFrame by multiple columns
print("Running snippet:")
print(snip_code)
print("\n--- OUTPUT ---")
result = run_code(snip_code)
print(result.results[0].content)
print(f"\nLatency: {result.latency_ms:.1f} ms")

In [ ]:
# Test run_code on list comprehension snippet
snip_code2 = SNIPPETS[8]["code"]  # List comprehension patterns
print("Running snippet:")
print(snip_code2[:200])
print("\n--- OUTPUT ---")
result2 = run_code(snip_code2)
print(result2.results[0].content)

In [ ]:
# Blocked import demonstration
blocked_snip = "import os\nprint(os.listdir('.'))"
print(f"Attempting: {blocked_snip}")
result3 = run_code(blocked_snip)
print(f"Result: {result3.results[0].content}")

## 11. Tool Call Trace Visualization

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

# ── Chart 1: Tools used per question ──
ax = axes[0, 0]
questions_short = [f"Q{i+1}: {a.question[:30]}..." for i, a in enumerate(assistant.history)]
tool_counts = [[tc.tool_name for tc in a.tool_calls] for a in assistant.history]

all_tool_names = ["search_docs", "search_snippets", "lookup_function",
                  "get_examples", "search_repo", "run_code"]
tool_colors = {
    "search_docs": "#3498db", "search_snippets": "#2ecc71",
    "lookup_function": "#e74c3c", "get_examples": "#f39c12",
    "search_repo": "#9b59b6", "run_code": "#1abc9c",
}

bottom = [0] * len(assistant.history)
for tool_name in all_tool_names:
    counts = [tc_list.count(tool_name) for tc_list in tool_counts]
    ax.bar(range(len(assistant.history)), counts, bottom=bottom,
           label=tool_name, color=tool_colors.get(tool_name, "#95a5a6"), alpha=0.8)
    bottom = [b + c for b, c in zip(bottom, counts)]

ax.set_xticks(range(len(assistant.history)))
ax.set_xticklabels([f"Q{i+1}" for i in range(len(assistant.history))])
ax.set_ylabel("Tool calls")
ax.set_title("Tools Called per Question")
ax.legend(fontsize=7, loc="upper right")

# ── Chart 2: Intent distribution ──
ax2 = axes[0, 1]
intent_map = {str(i): 0 for i in QueryIntent}
for a in assistant.history:
    intent_map[str(a.intent)] += 1
active = {k: v for k, v in intent_map.items() if v > 0}
intent_colors = ["#e74c3c", "#3498db", "#2ecc71", "#f39c12", "#9b59b6"]
ax2.pie(active.values(), labels=active.keys(), autopct="%1.0f%%",
        colors=intent_colors[:len(active)], startangle=90,
        textprops={"fontsize": 9})
ax2.set_title("Query Intent Distribution")

# ── Chart 3: Context pieces per question by type ──
ax3 = axes[1, 0]
# Re-run context to collect piece types
ctx_by_q = []
for a in assistant.history:
    type_counts = defaultdict(int)
    for tc in a.tool_calls:
        for p in tc.results:
            type_counts[p.source_type] += 1
    ctx_by_q.append(type_counts)

piece_types = ["doc", "snippet", "signature", "repo", "execution"]
piece_colors = ["#3498db", "#2ecc71", "#e74c3c", "#9b59b6", "#1abc9c"]
bottom2 = [0] * len(assistant.history)
for ptype, pcolor in zip(piece_types, piece_colors):
    vals = [cq.get(ptype, 0) for cq in ctx_by_q]
    ax3.bar(range(len(assistant.history)), vals, bottom=bottom2,
            label=ptype, color=pcolor, alpha=0.8)
    bottom2 = [b + v for b, v in zip(bottom2, vals)]
ax3.set_xticks(range(len(assistant.history)))
ax3.set_xticklabels([f"Q{i+1}" for i in range(len(assistant.history))])
ax3.set_ylabel("Context pieces")
ax3.set_title("Context Piece Types per Question")
ax3.legend(fontsize=8)

# ── Chart 4: Tool latency breakdown ──
ax4 = axes[1, 1]
tool_latencies = defaultdict(list)
for a in assistant.history:
    for tc in a.tool_calls:
        tool_latencies[tc.tool_name].append(tc.latency_ms)

if tool_latencies:
    t_labels = list(tool_latencies.keys())
    t_means = [sum(v) / len(v) for v in tool_latencies.values()]
    bars = ax4.barh(t_labels, t_means,
                    color=[tool_colors.get(t, "#95a5a6") for t in t_labels], alpha=0.8)
    ax4.set_xlabel("Avg latency (ms)")
    ax4.set_title("Average Tool Latency")
    for bar, val in zip(bars, t_means):
        ax4.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
                 f"{val:.1f}ms", va="center", fontsize=8)

plt.suptitle("Coding Assistant — Tool Trace Analytics", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## 12. Context Quality Analysis

In [ ]:
# Build a context quality DataFrame
rows = []
for i, answer in enumerate(assistant.history):
    for tc in answer.tool_calls:
        for piece in tc.results:
            rows.append({
                "question_idx": i + 1,
                "intent": str(answer.intent),
                "tool": tc.tool_name,
                "source_id": piece.source_id,
                "source_type": piece.source_type,
                "library": piece.library,
                "relevance_score": piece.score,
                "title": piece.title[:50],
            })

ctx_df = pd.DataFrame(rows)
print("Context quality by source type:")
print(ctx_df.groupby("source_type")["relevance_score"].agg(["count", "mean", "min", "max"]).round(3))

In [ ]:
print("\nTop 10 most-retrieved context pieces:")
top_ctx = (ctx_df.groupby(["source_id", "source_type", "library"])["relevance_score"]
           .agg(hits="count", avg_score="mean")
           .sort_values("hits", ascending=False)
           .head(10)
           .reset_index())
print(top_ctx.to_string(index=False))

## 13. Evaluation

We verify the agent's outputs against a simple **expected result** checklist.


In [ ]:
EXPECTED = {
    0: {  # Sort DataFrame
        "intent": QueryIntent.GENERATE,
        "has_code": True,
        "must_cite_any": ["pd-sort-01", "snip-df-sort"],
        "code_contains": "sort_values",
    },
    1: {  # __repr__ vs __str__
        "intent": QueryIntent.COMPARE,
        "has_code": True,
        "must_cite_any": ["py-dunder-01", "snip-repr-str"],
        "code_contains": "__repr__",
    },
    2: {  # async/await
        "intent": QueryIntent.GENERATE,
        "has_code": True,
        "must_cite_any": ["py-async-01", "snip-async"],
        "code_contains": "await",
    },
    3: {  # requests error
        "intent": QueryIntent.GENERATE,
        "has_code": True,
        "must_cite_any": ["req-01", "req-02", "snip-requests-error"],
        "code_contains": "raise_for_status",
    },
    4: {  # dataclasses default_factory
        "intent": QueryIntent.GENERATE,
        "has_code": True,
        "must_cite_any": ["py-dataclass-01", "snip-dataclass"],
        "code_contains": "default_factory",
    },
    5: {  # list comp vs generator
        "intent": QueryIntent.COMPARE,
        "has_code": True,
        "must_cite_any": ["py-generators-01", "snip-generator", "snip-list-comp"],
        "code_contains_any": ["yield", "for x in", "generator"],
    },
}

eval_rows = []
for idx, answer in enumerate(assistant.history):
    exp = EXPECTED.get(idx, {})
    results = {}

    results["intent_ok"] = exp.get("intent") is None or answer.intent == exp.get("intent")
    results["has_code"] = bool(answer.code_block.strip()) if exp.get("has_code") else True
    results["has_explanation"] = bool(answer.explanation.strip())
    results["has_citations"] = len(answer.citations) > 0

    cite_ok = True
    if "must_cite_any" in exp:
        cite_ok = any(cid in answer.citations for cid in exp["must_cite_any"])
    results["citation_ok"] = cite_ok

    code_ok = True
    if "code_contains" in exp:
        code_ok = exp["code_contains"] in answer.code_block
    if "code_contains_any" in exp:
        code_ok = any(k in answer.code_block for k in exp["code_contains_any"])
    results["code_ok"] = code_ok

    passed = all(results.values())

    print(f"Q{idx+1}: {answer.question[:50]}")
    for k, v in results.items():
        mark = "ok  " if v else "FAIL"
        print(f"     [{mark}] {k}")
    print(f"     → {'PASS' if passed else 'FAIL'}\n")

    eval_rows.append({"question": f"Q{idx+1}", "passed": passed, **results})

eval_df = pd.DataFrame(eval_rows)
pass_rate = eval_df["passed"].mean()
print(f"Overall pass rate: {eval_df['passed'].sum()}/{len(eval_df)} ({pass_rate:.0%})")

## 14. Session Statistics

In [ ]:
stats = assistant.stats()
print("SESSION SUMMARY")
print("=" * 50)
for k, v in stats.items():
    if isinstance(v, dict):
        print(f"{k}:")
        for kk, vv in v.items():
            if vv > 0:
                print(f"  {kk}: {vv}")
    else:
        print(f"{k}: {v}")

## 15. Save Experiment Log

In [ ]:
log = {
    "timestamp": datetime.now().isoformat(),
    "task": "coding_assistant",
    "questions_asked": len(assistant.history),
    "tools_available": list(TOOL_FN_MAP.keys()),
    "doc_fragments": len(DOC_FRAGMENTS),
    "code_snippets": len(SNIPPETS),
    "repo_files": len(REPO_FILES),
    "function_signatures": len(FUNCTION_SIGNATURES),
    "stats": stats,
    "eval_pass_rate": float(eval_df["passed"].mean()),
    "architecture": {
        "intent_classifier": "regex-rule-based (5 intents)",
        "tool_router": "intent-to-tool mapping",
        "context_builder": "multi-tool parallel retrieval + dedup + rank",
        "retrieval_method": "TF-IDF-style keyword overlap scoring",
        "answer_synthesizer": "template-based (LLM-ready)",
        "code_execution": "sandboxed exec() with import blocklist",
    },
}

log_path = ARTIFACT_DIR / "coding_assistant_log.json"
log_path.write_text(json.dumps(log, indent=2, default=str), encoding="utf-8")

history_path = ARTIFACT_DIR / "qa_history.json"
history_data = []
for a in assistant.history:
    history_data.append({
        "question": a.question,
        "intent": str(a.intent),
        "explanation": a.explanation[:200],
        "has_code": bool(a.code_block),
        "key_points_count": len(a.key_points),
        "citations": a.citations,
        "tool_calls": [{"tool": tc.tool_name, "results": len(tc.results), "latency_ms": tc.latency_ms}
                       for tc in a.tool_calls],
    })
history_path.write_text(json.dumps(history_data, indent=2), encoding="utf-8")

print(f"Saved: {log_path}")
print(f"Saved: {history_path}")

## 16. Key Takeaways

### What We Built

A **tool-assisted coding assistant** with five reusable retrieval/execution tools, an intent-aware tool router, a context builder with deduplication and re-ranking, and a structured answer synthesizer.

### Component Summary

| Component | Implementation | Production equivalent |
|---|---|---|
| Intent classifier | Regex rule patterns | Fine-tuned classifier or LLM prompt |
| Tool router | Intent → tool list mapping | LLM function-calling (OpenAI tool use) |
| `search_docs` | Keyword overlap scoring | BM25 / dense vector search |
| `search_snippets` | Tag + title matching | Embedding cosine similarity |
| `lookup_function` | Dictionary lookup | AST parsing + symbol index |
| `get_examples` | Topic fuzzy match | RAG over curated example corpus |
| `search_repo` | Content keyword overlap | Code-aware search (Sourcegraph, ripgrep) |
| `run_code` | `exec()` + stdout capture | Sandboxed container (Jupyter kernel) |
| Context builder | Score-ranked dedup pool | LLM context window packing |
| Answer synthesizer | Template sentences | LLM generation with context injection |

### Extending to a Real System

```python
# 1. Replace template synthesis with an LLM call:
def synthesize_answer_llm(question, intent, context, tool_calls):
    prompt = build_prompt(question, intent, context)
    response = openai.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        tools=TOOL_SCHEMAS,  # Enable function calling
    )
    return parse_answer(response)

# 2. Replace keyword scoring with embeddings:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
doc_vectors = embedder.encode([d["content"] for d in DOC_FRAGMENTS])

def search_docs_semantic(query: str, top_k=3):
    q_vec = embedder.encode([query])
    scores = cosine_similarity(q_vec, doc_vectors)[0]
    ...

# 3. Enable streaming output:
async def ask_stream(question: str):
    context, calls = build_context(question, ...)
    async for chunk in llm.stream(build_prompt(question, context)):
        yield chunk
```

### Key Design Principles
1. **Stateless tools** — each tool is a pure function, easy to test and swap
2. **Typed context pieces** — every tool returns the same `ContextPiece` type
3. **Intent-first routing** — reduces unnecessary tool calls
4. **Dedup + rerank** — prevents the same content from dominating the context window
5. **Separable layers** — retrieval, ranking, synthesis each independently improvable
6. **Sandboxed execution** — `run_code` blocks dangerous imports before `exec()`
